# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR² dataset using the [`mlcroissant`](https://pypi.org/project/mlcroissant/) library.

### Dataset Source
The dataset is defined by a Croissant schema and accessible at:
```
https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json
```

We will reference all dataset entities (record sets, fields, columns, etc.) by their `@id` identifiers, as per best practice.

In [ ]:
# Ensure mlcroissant is available in your environment
!pip install -q mlcroissant

## 1. Data Loading
Load dataset metadata and records from the schema using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Set the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Display dataset name and description from the metadata object (do not subscript directly)
print(f"{metadata.name}: {metadata.description}")
print(f"\nDataset Identifier: {getattr(metadata, 'identifier', 'N/A')}")

## 2. Data Overview
Review all available record sets and their fields, identified by their `@id` values.

In [ ]:
# List available record sets and their field @id's
print("Available record sets (by @id) and their fields:")
recordsets = []
for recordset in getattr(metadata, 'record_sets', []):
    rid = getattr(recordset, '@id', getattr(recordset, 'id', None))
    recordsets.append(rid)
    print(f"\nRecord Set @id: {rid}")
    if hasattr(recordset, 'fields'):
        for field in recordset.fields:
            fid = getattr(field, '@id', getattr(field, 'id', None))
            print(f"  Field @id: {fid}")

### Example: Print a preview of records from a record set
Pick a record set `@id` from above overview. Here we use the **first** record set for demonstration. All entities will be referenced by their `@id` in code and variables.

In [ ]:
# Get the first record set @id (if available)
first_recordset_id = recordsets[0] if recordsets else None
if first_recordset_id is not None:
    print(f"\nPreview of some records from record set @id={first_recordset_id}:")
    # Show up to 2 sample records for inspection
    for i, record in enumerate(dataset.records(record_set=first_recordset_id)):
        pprint.pprint(record)
        if i >= 1:
            break
else:
    print("No record sets defined in metadata.")

## 3. Data Extraction
Extract all records from one or more record sets into pandas DataFrames for analysis.

Record sets and DataFrame columns are referenced strictly by `@id`.

We will demonstrate extraction for *all* available record sets. You can modify the list as needed.

In [ ]:
# Compile full list of record set @id's
if not recordsets:
    raise RuntimeError("No record sets found.")

# Load all record sets into DataFrames
dataframes = {}
for record_set_id in recordsets:
    print(f"Loading data for record set @id={record_set_id} ...")
    records = list(dataset.records(record_set=record_set_id))
    if not records:
        print(f"  (No records found for @id={record_set_id})")
        continue
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"  Loaded {len(df)} records. Columns (by @id): {list(df.columns)}")

Let's inspect the data from the first available record set (by @id):

In [ ]:
if first_recordset_id in dataframes:
    print(f"Columns for record set @id={first_recordset_id}:")
    print(dataframes[first_recordset_id].columns.tolist())
    print("\nPreview:")
    display(dataframes[first_recordset_id].head())
else:
    print(f"No data available for record set @id={first_recordset_id}.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, referencing fields by their `@id`.

- Demonstrate filtering on a ***numeric*** field (by `@id`)
- Show normalization and grouping, referencing all columns by `@id`

*Note:* Please replace field and group `@id` values below with those that exist in your dataset after running the previous code cells!

In [ ]:
# Choose a record set and a numeric field by @id for EDA
# Replace these with actual @id's from your output above for meaningful analysis

record_set_id = first_recordset_id
# Example placeholders (replace with real ones):
numeric_field_id = None
group_field_id = None

if record_set_id in dataframes:
    df = dataframes[record_set_id].copy()
    # Try to automatically detect a numeric field
    for col in df.columns:
        # Check first N rows to detect if column is numeric
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break
    # Try to detect a non-numeric (categorical) field for grouping
    for col in df.columns:
        if col != numeric_field_id and df[col].dtype == object:
            group_field_id = col
            break
    if numeric_field_id:
        print(f"Numeric field for analysis (by @id): {numeric_field_id}")
        threshold = df[numeric_field_id].mean() if df[numeric_field_id].notnull().any() else 0
        # Filtering records where numeric_field > threshold
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records where {numeric_field_id} > threshold ({threshold:.2f if hasattr(threshold, 'is_integer') else threshold}): {len(filtered_df)} found.")
        print(filtered_df.head())

        # Normalizing the numeric field
        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, norm_col]].head())

        # Group by a group field (categorical)
        if group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame('mean_' + numeric_field_id)
            print(f"\nMean {numeric_field_id} grouped by {group_field_id}:")
            print(grouped_df.head())
        else:
            print("\nNo suitable group field (@id) detected for grouping in EDA.")
    else:
        print("No numeric fields available for EDA in this record set.")
else:
    print(f"No data found for record set @id={record_set_id}.")

## 5. Visualization
Visualize distributions or relationships between numeric and categorical fields using `matplotlib` or `seaborn`.

Again, all columns must be referenced by their `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_set_id in dataframes and numeric_field_id and group_field_id:
    plt.figure(figsize=(10, 5))
    sns.histplot(df[numeric_field_id].dropna(), bins=20, kde=True)
    plt.title(f"Distribution of '{numeric_field_id}'")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # Boxplot by group (if suitable)
    group_vals = df[group_field_id].unique()
    if len(group_vals) < 20:
        plt.figure(figsize=(12, 6))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("Cannot plot: missing numeric/group fields detected in dataset.")

## 6. Conclusion
In this notebook, we:
- Loaded and explored the FAIR² colorectal cancer survivors dataset using `mlcroissant` and strict `@id` referencing
- Listed all available record sets and fields by `@id`
- Extracted records to pandas DataFrames for programmatic analysis
- Demonstrated EDA and visualization referencing columns by their `@id`

This approach can be easily adapted for deeper domain-specific analysis, further subgroup analysis, or training ML models. For more information on the Croissant format or dataset schema, visit https://mlcommons.org/croissant/ and the dataset's [Croissant schema link](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json).